# NIFTY 50 — ML Model Training (Colab)

Upload `Stock-market-analysis-colab.zip`, then run all cells to train all models.

In [ ]:
# Step 1: Upload and extract the zip
from google.colab import files
import os

print('Upload Stock-market-analysis-colab.zip...')
uploaded = files.upload()

!unzip -qo Stock-market-analysis-colab.zip -d /content/Stock-market-analysis

os.chdir('/content/Stock-market-analysis')
print(f'Working directory: {os.getcwd()}')
print(f'Contents: {os.listdir(".")}')
print(f'ml/ contents: {os.listdir("ml")}')

In [ ]:
# Step 2: Install dependencies
!pip install -q scikit-learn pandas numpy pyarrow joblib

In [ ]:
# Step 3: Verify data exists
import os, pandas as pd

if os.path.exists('ml/data/processed/train.parquet'):
    print('Parquet files found:')
    for f in ['train', 'val', 'test']:
        df = pd.read_parquet(f'ml/data/processed/{f}.parquet')
        print(f'  {f}: {len(df):,} rows, {df["Date"].min().date()} to {df["Date"].max().date()}')
else:
    print('Parquet files not found. Running data pipeline...')
    !python -m ml.pipeline.run_pipeline

In [ ]:
# Step 4: Train ALL models (regression + classification + unsupervised)
import time
start = time.time()

!python -m ml.train.run_training

elapsed = time.time() - start
print(f'\n=============================')
print(f'Total training time: {elapsed/60:.1f} minutes')
print(f'=============================')

In [ ]:
# Step 5: View results
import json

with open('ml/models/metrics.json') as f:
    metrics = json.load(f)

print('=' * 70)
print('REGRESSION MODELS')
print('=' * 70)
for name, m in metrics.get('regression', {}).items():
    best = ' ★ BEST' if m.get('best') else ''
    v = m.get('val', {})
    print(f'  {name:25s} RMSE={v.get("rmse",0):10.4f}  MAE={v.get("mae",0):10.4f}  R²={v.get("r2",0):.6f}{best}')

print()
print('=' * 70)
print('CLASSIFICATION MODELS')
print('=' * 70)
for name, m in metrics.get('classification', {}).items():
    best = ' ★ BEST' if m.get('best') else ''
    v = m.get('val', {})
    print(f'  {name:25s} Acc={v.get("accuracy",0)*100:5.1f}%  F1={v.get("f1",0):.4f}  Prec={v.get("precision",0):.4f}  Rec={v.get("recall",0):.4f}{best}')

print()
print('=' * 70)
print('UNSUPERVISED')
print('=' * 70)
unsup = metrics.get('unsupervised', {})
if unsup:
    km = unsup.get('kmeans', {})
    pca = unsup.get('pca', {})
    print(f'  K-Means: {km.get("n_clusters", 0)} clusters, inertia={km.get("inertia", 0):.2f}')
    print(f'  PCA: explained variance={pca.get("total_explained_variance", 0)*100:.1f}%')

print()
print('=' * 70)
print('SAVED MODEL FILES')
print('=' * 70)
for subdir in ['regression', 'classification', 'clustering']:
    path = f'ml/models/{subdir}'
    if os.path.exists(path):
        pkl_files = [f for f in os.listdir(path) if f.endswith('.pkl')]
        print(f'  {subdir}/: {pkl_files}')

In [ ]:
# Step 6: Download trained models
!cd ml/models && zip -r /content/trained_models.zip \
    regression/ classification/ clustering/ \
    metrics.json scaler.pkl label_encoders.pkl

import os
size_mb = os.path.getsize('/content/trained_models.zip') / (1024*1024)
print(f'\ntrained_models.zip ready ({size_mb:.1f} MB)')
print('Downloading...')

from google.colab import files
files.download('/content/trained_models.zip')